In [ ]:
# import streamlit as st
import yfinance as yf
import pandas as pd
import plotly.express as px
import os
from dotenv import load_dotenv
load_dotenv()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# STEP 1: load financial ledger
df_ledger = pd.read_csv('/content/drive/MyDrive/Investment Forecasting/Streamlit App/financial_ledger.csv')
tickers_list = df_ledger["Ticker"].unique().tolist()

In [ ]:
tickers_list

In [ ]:
def load_price_data(tickers):
    df = yf.download(tickers, period="1y", auto_adjust=True)["Close"]
    df = df.reset_index()
    return df

In [ ]:
df_prices = load_price_data(tickers_list)

In [ ]:
df_prices_long = df_prices.melt(id_vars="Date", var_name="Ticker", value_name="Close")

In [ ]:
df_prices_long["Close"] = df_prices_long.groupby("Ticker")["Close"].ffill()

In [ ]:
# Assicurati che le date siano in formato datetime e senza timezone
df_prices_long['Date'] = pd.to_datetime(df_prices_long['Date']).dt.tz_localize(None)

# 1. ORDINAMENTO: Fondamentale per far sì che pct_change() confronti oggi con ieri
df_prices_long = df_prices_long.sort_values(by=["Ticker", "Date"])

# 2. RAGGRUPPAMENTO E CALCOLO:
# Calcola la variazione tra righe consecutive all'interno dello stesso Ticker
df_prices_long["Daily_Return_%"] = (
    df_prices_long.groupby("Ticker")["Close"]
    .pct_change() * 100
).round(2)

# 3. Volatility
# df_prices_long.groupby("Ticker")["Daily_Return_%"].transform(lambda x: x.rolling(30).std())

In [ ]:
### Volaitlity 30 gg
df_prices_long["Volatility_30d"] = df_prices_long.groupby("Ticker")["Daily_Return_%"].transform(lambda x: x.rolling(window=30).std())

In [ ]:
## Volatility
import numpy as np
df_prices_long["Volatility_Ann_%"] = (df_prices_long["Volatility_30d"] * np.sqrt(252)).round(2)

In [ ]:
df_prices_long

In [ ]:
df_prices_long["Date"] = pd.to_datetime(df_prices_long["Date"]).dt.tz_localize(None)
df_ledger["Date"] = pd.to_datetime(df_ledger["Date"]).dt.tz_localize(None)

In [ ]:
df_investments = df_ledger.groupby('Ticker')['Quantity'].sum().reset_index()
print(df_investments)

In [ ]:
df_merged = pd.merge(df_prices_long, df_investments, on=["Ticker"], how="left")

In [ ]:
### Market Value Or Position Value
df_merged["Market_Value"] = df_merged["Quantity"] * df_merged["Close"]

In [ ]:
### Total Position Value
df_merged["total_daily_val"] = df_merged.groupby("Date")["Market_Value"].transform("sum")

In [ ]:
### Exposure
df_merged["Exposure_%"] = (df_merged["Market_Value"] / df_merged["total_daily_val"] * 100).round(2)

# Agente

In [ ]:
# 1. Get the latest date available in the dataset
latest_date = df_merged["Date"].max()

# 2. Calculate the cutoff (30 days ago)
cutoff_date = latest_date - pd.Timedelta(days=30)

# 3. Filter the dataframe
df_last_30d = df_merged[df_merged["Date"] > cutoff_date].copy()

# 4. (Optional) Sort by Ticker and Date for consistency
df_last_30d = df_last_30d.sort_values(["Ticker", "Date"])

In [ ]:
df_last_30d  ###Input for agent

In [ ]:
df_last_30d  ###Input for agent

In [ ]:
# 1. Get the latest date available in the dataset
latest_date = df_merged["Date"].max()

# 2. Calculate the cutoff (30 days ago)
cutoff_date = latest_date

# 3. Filter the dataframe
last_date = df_merged[df_merged["Date"]== cutoff_date].copy()

# 4. (Optional) Sort by Ticker and Date for consistency
last_date = last_date.sort_values(["Ticker", "Date"]).reset_index().drop(columns=['index'])

In [ ]:
last_date

In [ ]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import AgentTool, FunctionTool, google_search
from google.genai import types

print("ADK components imported successfully.")

In [ ]:
import os
try:
    GOOGLE_API_KEY = "INSERT YOUR GEMINI API KEY"
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print(" Gemini API key setup complete.")
except Exception as e:
    print(
        f" Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

In [ ]:
retry_config=types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504], # Retry on these HTTP errors
)

In [ ]:
analyst_agent = Agent(
        name="StrategyAgent",
        model=Gemini(model="gemini-2.5-flash-lite"),
        instruction=f"""
        You are a Senior Portfolio Manager. Analyze the following quantitative data:
        {df_last_30d}

        Provide specific recommendations (BUY, SELL, HOLD, or REBALANCE) based on:
        1. Concentration Risk: Is Exposure % > 25% for a single asset?
        2. Efficiency: Is an asset showing High Volatility but Low/Negative ROI?
        3. Momentum: Are there consistent losses despite low volatility?

        Output Format: Ticker - ACTION - Technical justification (max 15 words).
        """,
    )


In [ ]:
runner = InMemoryRunner(agent=analyst_agent)

In [ ]:
response = await runner.run_debug("Provide concise strategic advice for my portfolio holdings.")


In [ ]:
today = df_merged["Date"].max()

purchase_list = []
for ticker in tickers_list:
    df_t_prices = df_prices_long[df_prices_long["Ticker"] == ticker].sort_values("Date")
    df_t_ledger = df_ledger[df_ledger["Ticker"] == ticker].sort_values("Date")
    if df_t_prices.empty or df_t_ledger.empty:
        continue
    merged = pd.merge_asof(df_t_ledger, df_t_prices, on="Date", direction="nearest")
    merged["Ticker"] = ticker
    merged = merged.rename(columns={"Close": "Purchase_Price"})
    merged = merged[["Ticker", "Date", "Quantity", "Purchase_Price"]]
    purchase_list.append(merged)

purchase_data = pd.concat(purchase_list, ignore_index=True)
purchase_data["Initial_Investment"] = purchase_data["Purchase_Price"] * purchase_data["Quantity"]

initial = purchase_data.groupby("Ticker").agg(
    Initial_Investment=("Initial_Investment", "sum"),
    Total_Quantity=("Quantity", "sum")
).reset_index()

# latest price = last available price per ticker (handles weekends/holidays)
latest_prices = df_prices_long.sort_values("Date").groupby("Ticker").last()[["Close"]].reset_index().rename(columns={"Close": "Latest_Price"})

summary = initial.merge(latest_prices, on="Ticker")
summary["Total_Investment"] = (summary["Latest_Price"] * summary["Total_Quantity"]).round(2)
summary["Unrealized Profit"] = (summary["Total_Investment"] - summary["Initial_Investment"]).round(2)
summary["ROI (%)"] = ((summary["Unrealized Profit"] / summary["Initial_Investment"]) * 100).round(2)

def get_price_on(days_ago):
    target = today - pd.Timedelta(days=days_ago)
    result = df_prices_long.groupby("Ticker").apply(
        lambda x: x[x["Date"] <= target]["Close"].iloc[-1] if not x[x["Date"] <= target].empty else None
    ).reset_index()
    result.columns = ["Ticker", f"Price_{days_ago}d"]
    return result

summary = summary.merge(get_price_on(1), on="Ticker")
summary = summary.merge(get_price_on(30), on="Ticker")
summary = summary.merge(get_price_on(365), on="Ticker")

summary["DoD (%)"] = ((summary["Latest_Price"] - summary["Price_1d"]) / summary["Price_1d"] * 100).round(2)
summary["MoM (%)"] = ((summary["Latest_Price"] - summary["Price_30d"]) / summary["Price_30d"] * 100).round(2)
summary["YoY (%)"] = ((summary["Latest_Price"] - summary["Price_365d"]) / summary["Price_365d"] * 100).round(2)

In [ ]:
summary

In [ ]:
# join with ledger on Ticker
df_merged = pd.merge(df_prices_long, df_ledger, on="Ticker", how="left")

# calculate investment value - quantity * close price
df_merged["Investment"] = df_merged["Quantity"] * df_merged["Close"]

# calculate daily return (% change day over day)
df_merged = df_merged.sort_values(["Ticker", "Date_x"])
df_merged["Return"] = df_merged.groupby("Ticker")["Investment"].pct_change() * 100

# STEP 4: calculate ROI, Unrealized Profit, DoD, MoM, YoY